# 面试问题：Rejection Sampling Fine-Tuning / Best-of-N 怎样构造训练数据？

**回答主线。** 对每个 prompt 从明确 policy revision 采样 N 个候选，先过硬安全/格式约束，再由 reward 或 verifier 排序，选择高质量且不过度重复的回答作为新 SFT 数据。N 增大通常提高观测到的最大 reward，但也放大 reward model 偏差、选择分布偏移和成本。

下面实现候选合同、Best-of-N、顺序统计、去重多样性、多目标门禁、assistant-only SFT loss、迭代版本和 paired bootstrap。代码只用 PyTorch 基础交叉熵，不调用 Trainer。


In [ ]:
import hashlib, json, math  # 导入本单元所需的依赖。
from dataclasses import dataclass  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。
import torch  # 导入本单元所需的依赖。

# NumPy 负责抽样评估，PyTorch 负责显式 token loss。
rng158 = np.random.default_rng(158)  # 计算并保存当前步骤的中间状态。
torch.manual_seed(158)  # 执行当前语句以推进本节示例。
assert torch.initial_seed() == 158  # 用受控断言验证关键不变量。
assert rng158.normal(size=4).shape == (4,)  # 用受控断言验证关键不变量。
assert torch.tensor([1, 2]).dtype == torch.int64  # 用受控断言验证关键不变量。


## 1. 候选必须绑定 prompt、policy、seed 与 sampling config

若只保存文本和 reward，无法判断候选来自哪个策略，也无法重放温度/top-p。相同 prompt 的 N 个候选必须作为一组切分，不能把兄弟样本泄漏到验证集。


In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Candidate158:  # 定义承载本节状态与行为的数据结构。
    prompt_id: str  # 执行当前语句以推进本节示例。
    text: str  # 执行当前语句以推进本节示例。
    policy_revision: str  # 执行当前语句以推进本节示例。
    seed: int  # 执行当前语句以推进本节示例。
    reward: float  # 执行当前语句以推进本节示例。
    safe: bool  # 执行当前语句以推进本节示例。
    format_ok: bool  # 执行当前语句以推进本节示例。

def validate_group158(candidates):  # 定义本节可复用的核心函数。
    # 同一组必须来自一个 prompt 和同一 policy revision，seed 不能重复。
    if len({c.prompt_id for c in candidates}) != 1 or len({c.policy_revision for c in candidates}) != 1:  # 按当前条件选择后续控制路径。
        raise ValueError("mixed candidate group")  # 遇到非法合同立即显式失败。
    if len({c.seed for c in candidates}) != len(candidates):  # 按当前条件选择后续控制路径。
        raise ValueError("duplicate sampling seed")  # 遇到非法合同立即显式失败。
    return True  # 返回当前分支计算出的结果。

group158 = [Candidate158("p1", f"answer {i}", "policy-v3", i, score, safe, True) for i, (score, safe) in enumerate([(0.2, True), (0.8, True), (1.2, False), (0.7, True)])]  # 计算并保存当前步骤的中间状态。
assert validate_group158(group158)  # 用受控断言验证关键不变量。
assert len(group158) == 4  # 用受控断言验证关键不变量。
assert {c.policy_revision for c in group158} == {"policy-v3"}  # 用受控断言验证关键不变量。


## 2. 硬约束先于 reward，非法高分候选不能入选

Safety、schema、引用或任务 verifier 属于准入条件；reward 只在可行集合内排序。全部失败时应记录 no-selection 并回采样/人审，而不是挑“最不坏”的非法答案。


In [ ]:
def best_of_n158(candidates):  # 定义本节可复用的核心函数。
    # 先过滤硬约束，再按 reward、较短长度和 seed 稳定排序。
    validate_group158(candidates)  # 执行当前语句以推进本节示例。
    feasible = [c for c in candidates if c.safe and c.format_ok]  # 计算并保存当前步骤的中间状态。
    if not feasible:  # 按当前条件选择后续控制路径。
        return None  # 返回当前分支计算出的结果。
    return max(feasible, key=lambda c: (c.reward, -len(c.text), -c.seed))  # 返回当前分支计算出的结果。

selected158 = best_of_n158(group158)  # 计算并保存当前步骤的中间状态。
assert selected158.seed == 1  # 用受控断言验证关键不变量。
assert selected158.reward == 0.8  # 用受控断言验证关键不变量。
assert best_of_n158([Candidate158("x", "bad", "v", 1, 9.0, False, True)]) is None  # 用受控断言验证关键不变量。


## 3. Best-of-N 收益递减且会放大 reward 噪声

从同一分布取更多样本，最大值单调上升，但边际增益下降；被选样本来自分布尾部，不再等同原 policy。应报告 reward 与独立人工质量的相关性，而不只画训练 reward。


In [ ]:
def expected_max158(n, trials=4000):  # 定义本节可复用的核心函数。
    # 用固定标准正态模拟顺序统计，只验证 Best-of-N 机制。
    samples = rng158.normal(size=(trials, n))  # 计算并保存当前步骤的中间状态。
    return float(np.max(samples, axis=1).mean())  # 返回当前分支计算出的结果。

maxima158 = [expected_max158(n) for n in (1, 2, 4, 8, 16)]  # 计算并保存当前步骤的中间状态。
gains158 = np.diff(maxima158)  # 计算并保存当前步骤的中间状态。
assert all(b > a for a, b in zip(maxima158, maxima158[1:]))  # 用受控断言验证关键不变量。
assert maxima158[-1] > 1.0  # 用受控断言验证关键不变量。
assert gains158[-1] < gains158[0]  # 用受控断言验证关键不变量。


## 4. 去重与 MMR 防止选集只剩 reward 模板

对每个 prompt 只取一个最高分仍可能让全局数据充满相同套话。下面用 token Jaccard 和 MMR 在 reward 与新颖性之间折中；生产可替换为 embedding，但阈值和版本仍要记录。


In [ ]:
def tokens158(text):  # 定义本节可复用的核心函数。
    return set(text.lower().split())  # 返回当前分支计算出的结果。

def jaccard158(a, b):  # 定义本节可复用的核心函数。
    # 空集合约定相似度为 1，避免两个空回答被视为多样。
    a, b = tokens158(a), tokens158(b)  # 计算并保存当前步骤的中间状态。
    return 1.0 if not (a or b) else len(a & b) / len(a | b)  # 返回当前分支计算出的结果。

def mmr_select158(candidates, limit, diversity_weight=0.4):  # 定义本节可复用的核心函数。
    remaining, chosen = list(candidates), []  # 计算并保存当前步骤的中间状态。
    while remaining and len(chosen) < limit:  # 在终止条件满足前持续推进状态。
        score = lambda c: c.reward - diversity_weight * max([jaccard158(c.text, x.text) for x in chosen] or [0.0])  # 计算并保存当前步骤的中间状态。
        best = max(remaining, key=lambda c: (score(c), -c.seed))  # 计算并保存当前步骤的中间状态。
        chosen.append(best); remaining.remove(best)  # 执行当前语句以推进本节示例。
    return chosen  # 返回当前分支计算出的结果。

diverse_group158 = [Candidate158("p", text, "v", i, reward, True, True) for i, (text, reward) in enumerate([("safe concise answer", 1.0), ("safe concise response", 0.99), ("alternative detailed reasoning", 0.9)])]  # 计算并保存当前步骤的中间状态。
diverse158 = mmr_select158(diverse_group158, 2, diversity_weight=0.5)  # 计算并保存当前步骤的中间状态。
assert diverse158[0].seed == 0  # 用受控断言验证关键不变量。
assert diverse158[1].seed == 2  # 用受控断言验证关键不变量。
assert jaccard158(diverse158[0].text, diverse158[1].text) == 0.0  # 用受控断言验证关键不变量。


## 5. 多目标选择先设 guardrail，再比较质量—成本

单一 reward 容易偏爱冗长或风格特征。可把安全/正确性设硬约束，在剩余候选上使用质量减长度成本，并按 slice 校准系数。Pareto 被支配候选不应获选。


In [ ]:
def constrained_score158(candidate, length_cost=0.01):  # 定义本节可复用的核心函数。
    # 硬约束失败返回负无穷，合法候选才计算软目标。
    if not (candidate.safe and candidate.format_ok):  # 按当前条件选择后续控制路径。
        return -np.inf  # 返回当前分支计算出的结果。
    return candidate.reward - length_cost * len(candidate.text.split())  # 返回当前分支计算出的结果。

short158 = Candidate158("p", "good answer", "v", 1, 0.9, True, True)  # 计算并保存当前步骤的中间状态。
long158 = Candidate158("p", "good answer with much unnecessary repeated filler text", "v", 2, 0.93, True, True)  # 计算并保存当前步骤的中间状态。
unsafe_high158 = Candidate158("p", "bad", "v", 3, 100.0, False, True)  # 计算并保存当前步骤的中间状态。
assert constrained_score158(short158) > constrained_score158(long158)  # 用受控断言验证关键不变量。
assert constrained_score158(unsafe_high158) == -np.inf  # 用受控断言验证关键不变量。
assert max([short158, long158, unsafe_high158], key=constrained_score158) == short158  # 用受控断言验证关键不变量。


## 6. 入选回答重新变成 assistant-only SFT 目标

RFT 最终仍可用 token cross-entropy 训练。Prompt、system 与 padding 不参与 loss，assistant response 参与；shift 后分子和分母只统计有效 target token。这里手写 gather 计算，不调用 Trainer。


In [ ]:
batch158, length158, vocab158 = 2, 7, 11  # 计算并保存当前步骤的中间状态。
logits158 = torch.randn(batch158, length158, vocab158, requires_grad=True)  # 计算并保存当前步骤的中间状态。
labels158 = torch.tensor([[1, 2, 3, 4, 5, 0, 0], [2, 3, 4, 5, 6, 7, 0]])  # 计算并保存当前步骤的中间状态。
assistant_mask158 = torch.tensor([[0, 0, 0, 1, 1, 0, 0], [0, 0, 1, 1, 1, 1, 0]], dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
# logits[t] 预测 labels[t+1]，同时把 mask 向左对齐到 target。
shifted_logp158 = torch.log_softmax(logits158[:, :-1], dim=-1)  # 计算并保存当前步骤的中间状态。
targets158 = labels158[:, 1:]  # 计算并保存当前步骤的中间状态。
valid158 = assistant_mask158[:, 1:]  # 计算并保存当前步骤的中间状态。
token_nll158 = -shifted_logp158.gather(-1, targets158.unsqueeze(-1)).squeeze(-1)  # 计算并保存当前步骤的中间状态。
loss158 = (token_nll158 * valid158).sum() / valid158.sum()  # 计算并保存当前步骤的中间状态。
loss158.backward()  # 执行当前语句以推进本节示例。
assert loss158.ndim == 0 and torch.isfinite(loss158)  # 用受控断言验证关键不变量。
assert int(valid158.sum()) == 6  # 用受控断言验证关键不变量。
assert logits158.grad is not None and torch.isfinite(logits158.grad).all()  # 用受控断言验证关键不变量。


## 7. 迭代 RFT 不能混用 stale policy 与 reward 版本

每轮数据是某个 policy 分布上的选择结果。升级 policy/reward 后应新建 dataset revision；继续混入旧候选需要明确重打分与权重，而不能假装同分布。


In [ ]:
def iteration_manifest158(round_id, policy_revision, reward_revision, candidates):  # 定义本节可复用的核心函数。
    # manifest 拒绝候选 policy 与声明 revision 不一致。
    if any(c.policy_revision != policy_revision for c in candidates):  # 按当前条件选择后续控制路径。
        raise ValueError("stale policy candidate")  # 遇到非法合同立即显式失败。
    record = {"round": round_id, "policy": policy_revision, "reward": reward_revision, "seeds": sorted(c.seed for c in candidates)}  # 计算并保存当前步骤的中间状态。
    payload = json.dumps(record, sort_keys=True, separators=(",", ":"))  # 计算并保存当前步骤的中间状态。
    return record, hashlib.sha256(payload.encode()).hexdigest()  # 返回当前分支计算出的结果。

manifest158, digest158 = iteration_manifest158(3, "policy-v3", "reward-v6", group158)  # 计算并保存当前步骤的中间状态。
assert manifest158["round"] == 3  # 用受控断言验证关键不变量。
assert manifest158["seeds"] == [0, 1, 2, 3]  # 用受控断言验证关键不变量。
assert len(digest158) == 64  # 用受控断言验证关键不变量。


## 8. Held-out paired eval 报差值置信区间和安全 slice

用同一 prompt 对比 base 与 RFT 可降低任务难度方差。Bootstrap 按 prompt 重采样，不按候选 token；若安全 slice 回退，即使总体均值上升也不能发布。


In [ ]:
def paired_bootstrap158(base, candidate, repeats=2000):  # 定义本节可复用的核心函数。
    # 每次重采样 prompt 索引，计算候选减基线的平均差。
    base, candidate = np.asarray(base), np.asarray(candidate)  # 计算并保存当前步骤的中间状态。
    if base.shape != candidate.shape:  # 按当前条件选择后续控制路径。
        raise ValueError("paired arrays required")  # 遇到非法合同立即显式失败。
    indices = rng158.integers(0, len(base), size=(repeats, len(base)))  # 计算并保存当前步骤的中间状态。
    diffs = (candidate - base)[indices].mean(axis=1)  # 计算并保存当前步骤的中间状态。
    return float((candidate - base).mean()), tuple(np.quantile(diffs, [0.025, 0.975]))  # 返回当前分支计算出的结果。

base_eval158 = np.array([0, 1, 0, 1, 0, 0, 1, 0], dtype=float)  # 计算并保存当前步骤的中间状态。
rft_eval158 = np.array([1, 1, 1, 1, 0, 1, 1, 1], dtype=float)  # 计算并保存当前步骤的中间状态。
delta158, ci158 = paired_bootstrap158(base_eval158, rft_eval158)  # 计算并保存当前步骤的中间状态。
assert delta158 > 0  # 用受控断言验证关键不变量。
assert ci158[0] <= delta158 <= ci158[1]  # 用受控断言验证关键不变量。
assert len(ci158) == 2  # 用受控断言验证关键不变量。


## 面试总结

- 每个 prompt 从固定 policy/sampling revision 采 N 个候选，硬 verifier 先过滤，reward 只在合法集合排序。
- Best-of-N 提高最大观测 reward，但收益递减并放大 reward 偏差；N、温度和成本必须一起报告。
- 全局选集要去重和保多样性，入选回答再按 assistant-only mask 做普通 SFT。
- 迭代数据绑定 policy/reward 版本，最终用 held-out paired eval、置信区间与安全 slice 发布。

延伸阅读：[Llama 2](https://arxiv.org/abs/2307.09288)、[InstructGPT](https://arxiv.org/abs/2203.02155)、[Scaling Laws for Reward Model Overoptimization](https://arxiv.org/abs/2210.10760)。
